# Pull flood events: Carolina Beach and Down East

This notebook downloads Sunny Day Flooding Project (SDFP) water levels for
the **Carolina Beach** and **Down East** sensors and builds each site's
flood event records with `poseidon_utils.EventTracker`. The same module is
used by the flood_event_QC app, so the outputs here match what the QC app
produces.

For each site it writes, to `data/flood_event_pull/<site>/` (git-ignored):

| Output | Contents |
|---|---|
| `abbr_flood_events.csv` | One row per flood event: start, end, duration, peak level |
| `flood_events.csv` | Every reading inside each event; the summary is on the event's last row |
| `sensor_outages.csv` | Data gaps longer than one hour, flagged if they start during a flood |
| `flood_plots/` | One water level plot per event |
| `raw_sensor_data.parquet` | The download, used to rebuild the outputs after hand edits |

A flood event starts when the road water level rises above 0.02 ft and
ends at the first reading below 0 ft. Events at different sensors that
overlap in time share an event number. Times are UTC; `_EST` columns use
a fixed UTC-5 offset.

**You need SDFP API credentials.** Put them in a git-ignored
`mysecrets.py` (in this folder or the repository root):

```python
API_USERNAME = "..."
API_PASSWORD = "..."
```

or leave it out and you will be prompted for them.

In [ ]:
import sys
from datetime import date
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from poseidon_utils import EventTracker, ImageHandler, load_site_config

REPO = Path.cwd().parent
OUT = REPO / "data" / "flood_event_pull"

## 1. Settings

Choose the date range to download (both ends inclusive). Each site maps
to an SDFP location name; the location expands to all of its sensors
(Carolina Beach: CB_01-CB_03, Down East: DE_01-DE_04).

In [ ]:
START_DATE = date(2025, 10, 1)
END_DATE = date(2025, 10, 31)

# Site config name -> SDFP location name
SITES = {
    "carolina_beach": "Carolina Beach",
    "down_east": "Down East",
}

# Days per API request; failed requests are retried one day at a time
CHUNK_DAYS = 30

In [ ]:
# Credentials: mysecrets.py if present, otherwise prompt
sys.path[:0] = [str(Path.cwd()), str(REPO)]
try:
    from mysecrets import API_PASSWORD, API_USERNAME
except ImportError:
    API_USERNAME = input("SDFP API username: ")
    API_PASSWORD = getpass("SDFP API password: ")

tracker = EventTracker((API_USERNAME, API_PASSWORD), START_DATE, END_DATE)

## 2. Download and build the event records

Each site is downloaded and processed in turn. Re-running is safe: new
events are merged into the existing CSVs, and rows already there (same
sensor and start time) are kept as they are, so hand edits survive.
Existing plots are not redrawn.

In [ ]:
raw_data = {}
for site, location in SITES.items():
    print(f"===== {location} =====")
    raw_data[site] = tracker.pull_data_gen_csvs_and_plots(
        location, chunk_days=CHUNK_DAYS, output_dir=str(OUT / site))

## 3. Summary

`flood days (EST)` counts every Eastern calendar day touched by an event.

In [ ]:
def read_if_exists(path):
    """Read a CSV, or return an empty table if it was not written."""
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


events, outages, summary = {}, {}, []
for site, location in SITES.items():
    folder = OUT / site
    events[site] = read_if_exists(folder / "abbr_flood_events.csv")
    outages[site] = read_if_exists(folder / "sensor_outages.csv")
    abbr = events[site]
    summary.append({
        "site": location,
        "readings": 0 if raw_data[site] is None else len(raw_data[site]),
        "events": abbr["flood_event"].nunique() if len(abbr) else 0,
        "sensor floods": len(abbr),
        "flood days (EST)": (EventTracker.num_of_flood_days(
            "EST", str(folder / "abbr_flood_events.csv")) if len(abbr) else 0),
        "peak (ft)": abbr["max_road_water_level_(ft)"].max() if len(abbr)
        else None,
        "outages": len(outages[site]),
    })
pd.DataFrame(summary).set_index("site")

### Events per site

One row per sensor flood. `outage` is `Yes` when a sensor outage started
during the event, so its peak or end time may be unreliable.

In [ ]:
columns = ["flood_event", "sensor_ID", "start_time_EST", "end_time_EST",
           "duration_(hours)", "max_road_water_level_(ft)", "outage"]
for site, location in SITES.items():
    print(f"===== {location}: {len(events[site])} sensor floods =====")
    if len(events[site]):
        display(events[site].reindex(columns=columns))

### Largest event at each site

In [ ]:
fig, axes = plt.subplots(1, len(SITES), figsize=(8 * len(SITES), 7))
for ax, (site, location) in zip(axes, SITES.items()):
    ax.set_axis_off()
    abbr = events[site]
    if abbr.empty:
        ax.set_title(f"{location}: no events")
        continue
    # Plot names get _2, _3, ... for a sensor's repeat floods within
    # one combined event (same numbering as EventTracker)
    repeat = abbr.groupby(["flood_event", "sensor_ID"]).cumcount() + 1
    i = abbr["max_road_water_level_(ft)"].idxmax()
    peak, suffix = abbr.loc[i], "" if repeat[i] == 1 else f"_{repeat[i]}"
    plot = (OUT / site / "flood_plots" /
            f"flood_event_{peak['flood_event']}_{peak['sensor_ID']}"
            f"{suffix}.png")
    if plot.exists():
        ax.imshow(plt.imread(plot))
    ax.set_title(f"{location}: event {peak['flood_event']} "
                 f"({peak['sensor_ID']}, {peak['max_road_water_level_(ft)']} ft)")
fig.tight_layout()

## 4. Optional: pull the event images

Copies every archive image taken during each event (plus a buffer) from
the SDFP image archive, and the daylight images taken during each sensor
outage so reviewers can check for missed flooding. Set `IMAGE_DRIVE` to
the archive root (the folder holding `<YYYY> Archive/`) to run it.

In [ ]:
IMAGE_DRIVE = None  # e.g. "/mnt/sdfp_images"
BUFFER_HOURS = 1

if IMAGE_DRIVE is None:
    print("Set IMAGE_DRIVE to pull images.")
else:
    handler = ImageHandler(IMAGE_DRIVE)
    for site, location in SITES.items():
        folder = OUT / site
        if (folder / "abbr_flood_events.csv").exists():
            n = handler.pull_flood_event_images(
                folder / "event_images", folder / "abbr_flood_events.csv",
                time_buffer_hours=BUFFER_HOURS)
            print(f"{location}: {n} event images")
        if (folder / "sensor_outages.csv").exists():
            n = handler.pull_outage_images(
                folder / "outage_images", folder / "sensor_outages.csv")
            print(f"{location}: {n} outage images")

## 5. Optional: rebuild after editing the events

If you merge, split, or delete events by editing a site's
`abbr_flood_events.csv` (for example after QC), run this for that site.
It recomputes each event's duration and peak from the cached download,
renumbers the events, rewrites `flood_events.csv`, re-flags outages, and
redraws every plot. Extra columns you add (such as notes) are kept.

In [ ]:
EDITED_SITE = None  # e.g. "carolina_beach"

if EDITED_SITE is not None:
    tracker.regenerate_outputs_from_csv(output_dir=str(OUT / EDITED_SITE))

## Next steps

* **Per-event folders** - `poseidon_deploy/naiads/run_flood_event_setup.py`
  splits `abbr_flood_events.csv`, `flood_events.csv`, and the pulled images
  into one folder per event for the POSEIDON processing chain.
* **Processing** - `poseidon_workflow.ipynb` runs the grid, rectification,
  depth, and roadway stages on one event folder.